# Calibration Diagnostics: PIT Analysis and Coverage

This notebook demonstrates calibration diagnostics for probabilistic forecasting models.

## What you'll learn:
- Understanding PIT (Probability Integral Transform) analysis
- Interpreting PIT histograms for calibration assessment
- Running KS tests for uniformity
- Creating coverage reliability diagrams
- Diagnosing and fixing calibration issues

## 1. Setup and Generate Sample Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import sys
sys.path.append('../..')

# Project imports
from uq.calibration import compute_pit, evaluate_pit_uniformity
from uq.metrics import compute_coverage_by_level

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

In [ ]:
# Generate different calibration scenarios
np.random.seed(42)
n_samples = 2000

# True values
y_true = np.random.randn(n_samples) * 0.01

# Scenario 1: Well-calibrated model (StudentT with correct df)
well_cal_loc = y_true + np.random.randn(n_samples) * 0.002
well_cal_scale = np.abs(np.random.gamma(2, 0.0015, n_samples))
well_cal_df = 4

# Scenario 2: Overconfident model (underestimates uncertainty)
overconf_loc = y_true + np.random.randn(n_samples) * 0.002
overconf_scale = np.abs(np.random.gamma(2, 0.0008, n_samples))  # Too narrow
overconf_df = 10  # Higher df = lighter tails

# Scenario 3: Underconfident model (overestimates uncertainty)
underconf_loc = y_true + np.random.randn(n_samples) * 0.002
underconf_scale = np.abs(np.random.gamma(2, 0.003, n_samples))  # Too wide
underconf_df = 4

# Scenario 4: Biased model with skewed errors
biased_loc = y_true + 0.002 + np.random.randn(n_samples) * 0.002
biased_scale = np.abs(np.random.gamma(2, 0.0015, n_samples))
biased_df = 4

print(f"Generated {n_samples} samples for 4 calibration scenarios")

## 2. Understanding PIT (Probability Integral Transform)

PIT transforms observations using the predictive CDF. For well-calibrated models, PIT values should be uniformly distributed on [0,1].

In [ ]:
def compute_pit_studentt(y_true, loc, scale, df):
    """Compute PIT values for Student-t distribution."""
    # Standardize observations
    z = (y_true - loc) / scale
    
    # Compute CDF values (PIT)
    pit = stats.t.cdf(z, df=df)
    
    return pit

# Compute PIT for each scenario
scenarios = {
    'Well Calibrated': compute_pit_studentt(y_true, well_cal_loc, well_cal_scale, well_cal_df),
    'Overconfident': compute_pit_studentt(y_true, overconf_loc, overconf_scale, overconf_df),
    'Underconfident': compute_pit_studentt(y_true, underconf_loc, underconf_scale, underconf_df),
    'Biased': compute_pit_studentt(y_true, biased_loc, biased_scale, biased_df)
}

print("PIT values computed for all scenarios")

## 3. PIT Histogram Analysis

In [ ]:
# Create PIT histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, pit_values) in enumerate(scenarios.items()):
    ax = axes[idx]
    
    # Create histogram with 20 bins (standard for PIT)
    counts, bins, _ = ax.hist(pit_values, bins=20, density=True, 
                              alpha=0.7, color='blue', edgecolor='black')
    
    # Add uniform reference line
    ax.axhline(y=1.0, color='red', linestyle='--', label='Uniform')
    
    # Add confidence bands for uniformity (95% level)
    n_bins = 20
    expected = n_samples / n_bins
    std_dev = np.sqrt(expected * (1 - 1/n_bins))
    conf_lower = (expected - 1.96 * std_dev) / n_samples * n_bins
    conf_upper = (expected + 1.96 * std_dev) / n_samples * n_bins
    
    ax.axhline(y=conf_lower, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(y=conf_upper, color='gray', linestyle=':', alpha=0.5)
    ax.fill_between([0, 1], conf_lower, conf_upper, alpha=0.1, color='gray')
    
    # Compute KS test
    ks_stat, ks_pval = stats.kstest(pit_values, 'uniform')
    
    # Interpretation
    if ks_pval > 0.05:
        cal_status = "Well calibrated ✓"
        color = 'green'
    else:
        # Determine type of miscalibration
        pit_mean = np.mean(pit_values)
        pit_std = np.std(pit_values)
        
        if pit_std < 0.28:  # Theoretical std of uniform is ~0.289
            cal_status = "Overconfident ✗"
            color = 'red'
        elif pit_std > 0.31:
            cal_status = "Underconfident ✗"
            color = 'orange'
        else:
            cal_status = "Miscalibrated ✗"
            color = 'red'
    
    ax.set_title(f'{name}\nKS p-value: {ks_pval:.3f} - {cal_status}', 
                color=color, fontweight='bold')
    ax.set_xlabel('PIT Value')
    ax.set_ylabel('Density')
    ax.set_xlim(0, 1)
    ax.legend(loc='upper right')

plt.suptitle('PIT Histogram Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation Guide:")
print("- Uniform (flat): Well calibrated")
print("- U-shaped: Overconfident (intervals too narrow)")
print("- ∩-shaped: Underconfident (intervals too wide)")
print("- Skewed: Biased predictions")

## 4. Detailed PIT Statistics

In [ ]:
# Compute detailed PIT statistics
print("=" * 70)
print("PIT STATISTICS ANALYSIS")
print("=" * 70)

for name, pit_values in scenarios.items():
    print(f"\n{name}:")
    
    # Basic statistics
    print(f"  Mean:     {np.mean(pit_values):.3f} (target: 0.500)")
    print(f"  Median:   {np.median(pit_values):.3f} (target: 0.500)")
    print(f"  Std Dev:  {np.std(pit_values):.3f} (target: 0.289)")
    
    # Quantiles
    q25 = np.percentile(pit_values, 25)
    q75 = np.percentile(pit_values, 75)
    print(f"  Q25:      {q25:.3f} (target: 0.250)")
    print(f"  Q75:      {q75:.3f} (target: 0.750)")
    
    # Statistical tests
    ks_stat, ks_pval = stats.kstest(pit_values, 'uniform')
    ad_stat, ad_crit, ad_sig = stats.anderson(pit_values, dist='uniform')
    
    print(f"\n  KS Test:")
    print(f"    Statistic: {ks_stat:.4f}")
    print(f"    P-value:   {ks_pval:.4f}")
    print(f"    Result:    {'Pass ✓' if ks_pval > 0.05 else 'Fail ✗'}")
    
    print(f"\n  Anderson-Darling Test:")
    print(f"    Statistic: {ad_stat:.4f}")
    print(f"    Critical (5%): {ad_crit[2]:.4f}")
    print(f"    Result:    {'Pass ✓' if ad_stat < ad_crit[2] else 'Fail ✗'}")

## 5. Coverage Analysis Across Multiple Levels

In [ ]:
# Compute coverage at multiple levels
coverage_levels = np.arange(10, 100, 5)
coverage_results = {}

for name in ['Well Calibrated', 'Overconfident', 'Underconfident', 'Biased']:
    if name == 'Well Calibrated':
        loc, scale, df = well_cal_loc, well_cal_scale, well_cal_df
    elif name == 'Overconfident':
        loc, scale, df = overconf_loc, overconf_scale, overconf_df
    elif name == 'Underconfident':
        loc, scale, df = underconf_loc, underconf_scale, underconf_df
    else:
        loc, scale, df = biased_loc, biased_scale, biased_df
    
    empirical_coverage = []
    
    for level in coverage_levels:
        # Compute prediction intervals
        alpha = (100 - level) / 100
        lower = stats.t.ppf(alpha/2, df=df, loc=loc, scale=scale)
        upper = stats.t.ppf(1-alpha/2, df=df, loc=loc, scale=scale)
        
        # Check coverage
        covered = (y_true >= lower) & (y_true <= upper)
        empirical_coverage.append(np.mean(covered) * 100)
    
    coverage_results[name] = empirical_coverage

print("Coverage computed for all levels and scenarios")

In [ ]:
# Create coverage reliability diagram
fig, ax = plt.subplots(figsize=(10, 8))

colors = {'Well Calibrated': 'green', 'Overconfident': 'red', 
          'Underconfident': 'orange', 'Biased': 'purple'}

for name, coverage in coverage_results.items():
    ax.plot(coverage_levels, coverage, 'o-', label=name, 
            color=colors[name], alpha=0.7, linewidth=2)

# Add perfect calibration line
ax.plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Perfect calibration')

# Add tolerance bands
ax.fill_between([0, 100], [-2, 98], [2, 102], 
                alpha=0.1, color='gray', label='±2% tolerance')

# Highlight key levels
for level in [80, 90, 95]:
    ax.axvline(x=level, color='gray', linestyle=':', alpha=0.3)
    ax.text(level, 5, f'{level}%', ha='center', fontsize=9, color='gray')

ax.set_xlabel('Nominal Coverage (%)', fontsize=12)
ax.set_ylabel('Empirical Coverage (%)', fontsize=12)
ax.set_title('Coverage Reliability Diagram', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 6. Quantile-Based PIT for Models Without Distributional Assumptions

In [ ]:
# For quantile models, we approximate PIT using quantile predictions
# Generate synthetic quantile predictions
np.random.seed(42)

# Dense quantile grid (1-99 percentiles)
quantile_levels = np.arange(0.01, 1.00, 0.01)
n_quantiles = len(quantile_levels)

# Generate quantile predictions for a well-calibrated quantile model
quantile_preds = np.zeros((n_samples, n_quantiles))

for i, q in enumerate(quantile_levels):
    # For demonstration, use normal quantiles with some noise
    z = stats.norm.ppf(q)
    quantile_preds[:, i] = well_cal_loc + z * well_cal_scale * (1 + np.random.randn(n_samples) * 0.05)

print(f"Generated {n_quantiles} quantile predictions for PIT approximation")

In [ ]:
def compute_pit_from_quantiles(y_true, quantile_preds, quantile_levels):
    """Approximate PIT using quantile predictions."""
    pit_values = []
    
    for i in range(len(y_true)):
        # Find where y_true falls in the quantile predictions
        # Linear interpolation between quantiles
        pit = np.interp(y_true[i], quantile_preds[i], quantile_levels)
        pit_values.append(pit)
    
    return np.array(pit_values)

# Compute PIT from quantiles
pit_quantile = compute_pit_from_quantiles(y_true, quantile_preds, quantile_levels)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PIT histogram
ax = axes[0]
ax.hist(pit_quantile, bins=20, density=True, alpha=0.7, 
        color='blue', edgecolor='black')
ax.axhline(y=1.0, color='red', linestyle='--', label='Uniform')
ax.set_title('PIT from Quantile Predictions')
ax.set_xlabel('PIT Value')
ax.set_ylabel('Density')
ax.legend()

# Q-Q plot
ax = axes[1]
stats.probplot(pit_quantile, dist="uniform", plot=ax)
ax.set_title('Q-Q Plot: PIT vs Uniform')

plt.tight_layout()
plt.show()

# KS test
ks_stat, ks_pval = stats.kstest(pit_quantile, 'uniform')
print(f"\nQuantile-based PIT Analysis:")
print(f"  KS statistic: {ks_stat:.4f}")
print(f"  KS p-value: {ks_pval:.4f}")
print(f"  Result: {'Well calibrated ✓' if ks_pval > 0.05 else 'Miscalibrated ✗'}")

## 7. Diagnosing and Fixing Calibration Issues

In [ ]:
# Diagnostic flowchart
print("=" * 70)
print("CALIBRATION DIAGNOSTIC GUIDE")
print("=" * 70)

diagnostic_guide = """
1. OVERCONFIDENT MODEL (U-shaped PIT, undercoverage)
   Symptoms:
   - PIT histogram is U-shaped
   - Coverage below nominal levels
   - Intervals too narrow
   
   Fixes:
   - Increase dropout rate in neural networks
   - Use heavier-tailed distribution (lower df for StudentT)
   - Add ensemble diversity
   - Increase regularization (L2, weight decay)
   - Use temperature scaling for calibration

2. UNDERCONFIDENT MODEL (∩-shaped PIT, overcoverage)
   Symptoms:
   - PIT histogram is inverse-U shaped
   - Coverage above nominal levels
   - Intervals too wide
   
   Fixes:
   - Reduce dropout rate
   - Use lighter-tailed distribution (higher df for StudentT)
   - Reduce ensemble size or diversity
   - Decrease regularization
   - Train for more epochs (may be undertrained)

3. BIASED MODEL (skewed PIT)
   Symptoms:
   - PIT histogram is skewed left or right
   - Systematic over/under prediction
   - Mean PIT ≠ 0.5
   
   Fixes:
   - Check for data leakage
   - Verify feature shift(1) is applied
   - Add bias correction term
   - Check for non-stationarity in data
   - Consider detrending or differencing

4. HEAVY-TAILED ERRORS (PIT concentrated at extremes)
   Symptoms:
   - High density at PIT values near 0 and 1
   - Poor coverage at all levels
   - Model struggles with outliers
   
   Fixes:
   - Use StudentT distribution instead of Gaussian
   - Implement robust loss functions (Huber, Tukey)
   - Add outlier detection and handling
   - Consider mixture models
   - Use quantile regression (MQLoss)
"""

print(diagnostic_guide)

## 8. Calibration Improvement Example

In [ ]:
# Demonstrate calibration improvement through temperature scaling
def temperature_scaling(loc, scale, temperature):
    """Apply temperature scaling to improve calibration."""
    # Temperature > 1: increases uncertainty (fixes overconfidence)
    # Temperature < 1: decreases uncertainty (fixes underconfidence)
    return loc, scale * temperature

# Find optimal temperature for overconfident model
temperatures = np.linspace(0.8, 2.0, 50)
best_temp = 1.0
best_ks_pval = 0

for temp in temperatures:
    _, scaled_scale = temperature_scaling(overconf_loc, overconf_scale, temp)
    pit_temp = compute_pit_studentt(y_true, overconf_loc, scaled_scale, overconf_df)
    ks_stat, ks_pval = stats.kstest(pit_temp, 'uniform')
    
    if ks_pval > best_ks_pval:
        best_ks_pval = ks_pval
        best_temp = temp

print(f"Optimal temperature for overconfident model: {best_temp:.3f}")
print(f"KS p-value improved from {stats.kstest(scenarios['Overconfident'], 'uniform')[1]:.4f} to {best_ks_pval:.4f}")

In [ ]:
# Visualize improvement
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original overconfident
ax = axes[0]
ax.hist(scenarios['Overconfident'], bins=20, density=True, 
        alpha=0.7, color='red', edgecolor='black')
ax.axhline(y=1.0, color='black', linestyle='--')
ax.set_title('Original (Overconfident)', color='red')
ax.set_xlabel('PIT Value')
ax.set_ylabel('Density')

# After temperature scaling
ax = axes[1]
_, scaled_scale = temperature_scaling(overconf_loc, overconf_scale, best_temp)
pit_improved = compute_pit_studentt(y_true, overconf_loc, scaled_scale, overconf_df)
ax.hist(pit_improved, bins=20, density=True, 
        alpha=0.7, color='green', edgecolor='black')
ax.axhline(y=1.0, color='black', linestyle='--')
ax.set_title(f'After Temperature Scaling (T={best_temp:.2f})', color='green')
ax.set_xlabel('PIT Value')

# Coverage comparison
ax = axes[2]
levels = [80, 90, 95]
original_cov = []
improved_cov = []

for level in levels:
    alpha = (100 - level) / 100
    
    # Original
    lower_orig = stats.t.ppf(alpha/2, df=overconf_df, loc=overconf_loc, scale=overconf_scale)
    upper_orig = stats.t.ppf(1-alpha/2, df=overconf_df, loc=overconf_loc, scale=overconf_scale)
    cov_orig = np.mean((y_true >= lower_orig) & (y_true <= upper_orig)) * 100
    original_cov.append(cov_orig)
    
    # Improved
    lower_imp = stats.t.ppf(alpha/2, df=overconf_df, loc=overconf_loc, scale=scaled_scale)
    upper_imp = stats.t.ppf(1-alpha/2, df=overconf_df, loc=overconf_loc, scale=scaled_scale)
    cov_imp = np.mean((y_true >= lower_imp) & (y_true <= upper_imp)) * 100
    improved_cov.append(cov_imp)

x = np.arange(len(levels))
width = 0.35

bars1 = ax.bar(x - width/2, original_cov, width, label='Original', color='red', alpha=0.7)
bars2 = ax.bar(x + width/2, improved_cov, width, label='Calibrated', color='green', alpha=0.7)

# Add nominal levels
for i, level in enumerate(levels):
    ax.axhline(y=level, xmin=i/3-0.1, xmax=i/3+0.2, 
               color='black', linestyle='--', alpha=0.5)

ax.set_xlabel('Coverage Level')
ax.set_ylabel('Empirical Coverage (%)')
ax.set_title('Coverage Improvement')
ax.set_xticks(x)
ax.set_xticklabels([f'{l}%' for l in levels])
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Calibration Improvement via Temperature Scaling', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Conformal Prediction Note

For models using conformal prediction, PIT analysis is not applicable because conformal intervals are constructed differently.

In [ ]:
print("=" * 70)
print("CONFORMAL PREDICTION AND CALIBRATION")
print("=" * 70)

conformal_note = """
Conformal Prediction provides a different approach to uncertainty quantification:

1. GUARANTEED COVERAGE
   - Conformal methods provide finite-sample coverage guarantees
   - Coverage is exact (up to discretization) under exchangeability
   - No distributional assumptions required

2. WHY PIT DOESN'T APPLY
   - Conformal intervals are not based on a predictive distribution
   - They are constructed from nonconformity scores
   - PIT requires a CDF, which conformal methods don't provide

3. CALIBRATION ASSESSMENT FOR CONFORMAL
   - Check empirical coverage directly
   - Verify coverage holds across different data subsets
   - Monitor interval width stability
   - Test conditional coverage (harder to achieve)

4. WHEN TO USE CONFORMAL
   - When coverage guarantees are critical
   - When distribution assumptions are questionable
   - For high-stakes applications requiring reliability
   - When you have sufficient calibration data

5. NEURALFORECAST IMPLEMENTATION
   ```python
   from neuralforecast.utils import PredictionIntervals
   
   # Enable conformal prediction
   nf = NeuralForecast(
       models=models,
       freq='15min'
   )
   
   # Fit with prediction intervals
   nf.fit(df, prediction_intervals=PredictionIntervals(n_windows=10))
   ```

Note: In our CV runner, set use_conformal=True to enable conformal intervals.
"""

print(conformal_note)

## Summary

### Key Takeaways:

1. **PIT Analysis**
   - Transforms observations using predictive CDF
   - Should be uniform [0,1] for calibrated models
   - Visual patterns indicate type of miscalibration

2. **Calibration Patterns**
   - U-shaped PIT → Overconfident (narrow intervals)
   - ∩-shaped PIT → Underconfident (wide intervals)
   - Skewed PIT → Biased predictions
   - Uniform PIT → Well calibrated

3. **Statistical Tests**
   - KS test: Overall uniformity (p > 0.05 is good)
   - Anderson-Darling: More sensitive to tails
   - Visual inspection often most informative

4. **Coverage Analysis**
   - Should match nominal levels within ±2%
   - Reliability diagram shows calibration across all levels
   - Systematic deviations indicate calibration issues

5. **Calibration Fixes**
   - Temperature scaling: Simple post-hoc adjustment
   - Distribution choice: StudentT for heavy tails
   - Model adjustments: Dropout, regularization
   - Conformal prediction: When guarantees needed

### Next Steps:

- Apply PIT analysis to your CV results
- Identify calibration issues in your models
- Use appropriate fixes based on diagnosis
- Consider conformal prediction for critical applications
- See `04_model_selection.ipynb` for using calibration in selection